In [ ]:
import pandas as pd
import missingno as msno
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor

In [ ]:
df=pd.read_csv("Data/variables_2024.csv")
df.head()

In [ ]:
df.shape

In [ ]:
df.describe()

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
porcentaje=df.isna().mean()*100
print(porcentaje)

In [ ]:
msno.matrix(df)  # para visualizar los nulos. hay una gran cantidad en varias columnas. 
# primero intento imputar usando las fórmulas y otras variables.

In [ ]:
df.columns

## Identificar outliers en las variables indicadas

CÓDIGO CNAE 2009

In [ ]:
Q1 = df["Código primario CNAE 2009"].quantile(0.25)
Q3 = df["Código primario CNAE 2009"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(lower_bound)
print(upper_bound)

In [ ]:
outliers = ((df["Código primario CNAE 2009"] < lower_bound) | (df["Código primario CNAE 2009"] > upper_bound))
df["Código primario CNAE 2009"].hist()

In [ ]:
outliers_valores = df["Código primario CNAE 2009"][
    (df["Código primario CNAE 2009"] < Q1 - 1.5 * IQR) | 
    (df["Código primario CNAE 2009"] > Q3 + 1.5 * IQR)
]

print(outliers_valores)

# no tiene sentido imputar porque son códigos sobre a qué se dedica cada empresa

FECHA CONSTITUCION

In [ ]:
df["año"] = pd.to_datetime(df["Fecha constitución"], dayfirst=True, errors="coerce").dt.year

In [ ]:
df["año"].isna().value_counts()

In [ ]:
df = df.dropna(subset=["año"])

In [ ]:
df["año"]= pd.to_numeric(df["año"])

In [ ]:
# IQR
q1 = df["año"].quantile(0.25)
q3 = df["año"].quantile(0.75)
iqr = q3 - q1


limite_inferior = q1 - 1.5 * iqr
limite_superior = q3 + 1.5 * iqr

In [ ]:
mediana = df['año'].median()
outliers = ((df['año'] < limite_inferior) | (df['año'] > limite_superior))

In [ ]:
outliers

In [ ]:
df['año'].hist(bins=10)

NUMERO EMPLEADOS

In [ ]:
df["Número empleados\nÚlt. año disp."] = df["Número empleados\nÚlt. año disp."].astype('Int64') 
# cambio esto porque estaba en float

Q1 = df["Número empleados\nÚlt. año disp."].quantile(0.25)
Q3 = df["Número empleados\nÚlt. año disp."].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR
print(lower_bound)
print(upper_bound)

In [ ]:
outliers = ((df["Número empleados\nÚlt. año disp."] < lower_bound) | (df["Número empleados\nÚlt. año disp."] > upper_bound))
df["Número empleados\nÚlt. año disp."].hist(bins=50, log=True)  #pongo log porque hay outliers grandes y sino no se ve bien la diferencia graficamente
plt.xlim(0, 5000)
plt.xlabel("Nº de empelados")
plt.ylabel("Frecuencia")
plt.show()

In [ ]:
outliers_valores = df["Número empleados\nÚlt. año disp."][
    (df["Número empleados\nÚlt. año disp."] < Q1 - 1.5 * IQR) | 
    (df["Número empleados\nÚlt. año disp."] > Q3 + 1.5 * IQR)
]

print(outliers_valores.max())
df.loc[df["Número empleados\nÚlt. año disp."] == 28975]

In [ ]:
df[df["Número empleados\nÚlt. año disp."] <= 0]  #no hay valores negativos ni igual a 0

## FORMULAS PARA NULOS

In [ ]:
#ACTIVO CIRCULANTE
activo="Activo circulante\nmil EUR\nÚlt. año disp."
col1="Existencias\nmil EUR\nÚlt. año disp."
col2="Deudores\nmil EUR\nÚlt. año disp."
col3="Otros activos líquidos\nmil EUR\nÚlt. año disp."
col4="Tesorería\nmil EUR\nÚlt. año disp."

variables= [col1,col2,col3,col4,activo]

for i in df.index:

    if df.loc[i, variables].isna().sum() > 1:
        continue
    #uso pd.isna porque con el loc me da un valor unico y isna no se puede con eso
    if pd.isna(df.loc[i, col1]):
        df.loc[i, col1] = df.loc[i, activo] - df.loc[i, col2] - df.loc[i, col3] - df.loc[i, col4]

    if pd.isna(df.loc[i, col2]):
        df.loc[i, col2] = df.loc[i, activo] - df.loc[i, col1] - df.loc[i, col3] - df.loc[i, col4]

    if pd.isna(df.loc[i, col3]):
        df.loc[i, col3] = df.loc[i, activo] - df.loc[i, col1] - df.loc[i, col2] - df.loc[i, col4]

    if pd.isna(df.loc[i, col4]):
        df.loc[i, col4] = df.loc[i, activo] - df.loc[i, col1] - df.loc[i, col2] - df.loc[i, col3]

    if pd.isna(df.loc[i, activo]):
        df.loc[i, activo] = df.loc[i, col1] + df.loc[i, col2] + df.loc[i, col3] + df.loc[i, col4]

df.isna().sum()  # ahora hay menos nulos en existencias



In [ ]:
#INMOVILIZADO
activo="Inmovilizado\nmil EUR\nÚlt. año disp."
col1="Inmovilizado inmaterial\nmil EUR\nÚlt. año disp."
col2="Inmovilizado material\nmil EUR\nÚlt. año disp."
col3="Otros activos fijos\nmil EUR\nÚlt. año disp."


variables= [col1,col2,col3,activo]

for i in df.index:

    if df.loc[i, variables].isna().sum() > 1:
        continue
    #uso pd.isna porque con el loc me da un valor unico y isna no se puede con eso
    if pd.isna(df.loc[i, col1]):
        df.loc[i, col1] = df.loc[i, activo] - df.loc[i, col2] - df.loc[i, col3]

    if pd.isna(df.loc[i, col2]):
        df.loc[i, col2] = df.loc[i, activo] - df.loc[i, col1] - df.loc[i, col3] 

    if pd.isna(df.loc[i, col3]):
        df.loc[i, col3] = df.loc[i, activo] - df.loc[i, col1] - df.loc[i, col2] 

    if pd.isna(df.loc[i, activo]):
        df.loc[i, activo] = df.loc[i, col1] + df.loc[i, col2] + df.loc[i, col3] 



In [ ]:
# PASIVO Y CAPITAL PROPIO
activo="Total pasivo y capital propio\nmil EUR\nÚlt. año disp."
col1="Total activo\nmil EUR\nÚlt. año disp."
col2="Capital suscrito\nmil EUR\nÚlt. año disp."


variables= [col1,col2,activo]

for i in df.index:

    if df.loc[i, variables].isna().sum() > 1:
        continue
    #uso pd.isna porque con el loc me da un valor unico y isna no se puede con eso
    if pd.isna(df.loc[i, col1]):
        df.loc[i, col1] = df.loc[i, activo] + df.loc[i, col2] 

    if pd.isna(df.loc[i, col2]):
        df.loc[i, col2] = df.loc[i, col1]-df.loc[i, activo]  

    if pd.isna(df.loc[i, activo]):
        df.loc[i, activo] = df.loc[i, col1] - df.loc[i, col2] 



In [ ]:

#Total activo:
total_activo="Total activo\nmil EUR\nÚlt. año disp."
col1="Inmovilizado\nmil EUR\nÚlt. año disp."
col2="Activo circulante\nmil EUR\nÚlt. año disp."

variables= [col1,col2,total_activo]
for i in df.index:

    if df.loc[i, variables].isna().sum() > 1:
        continue
    if pd.isna(df.loc[i, col1]):
        df.loc[i, col1] = df.loc[i, total_activo] - df.loc[i, col2]

    if pd.isna(df.loc[i, col2]):
        df.loc[i, col2] = df.loc[i, total_activo] - df.loc[i, col1]

    if pd.isna(df.loc[i, total_activo]):
        df.loc[i, total_activo] = df.loc[i, col1] + df.loc[i, col2]


In [ ]:

#Fondos PROPIOS:
activo="Fondos propios\nmil EUR\nÚlt. año disp."
col1="Capital suscrito\nmil EUR\nÚlt. año disp."
col2="Otros fondos propios\nmil EUR\nÚlt. año disp."

variables= [col1,col2,activo]
for i in df.index:

    if df.loc[i, variables].isna().sum() > 1:
        continue
    
    if pd.isna(df.loc[i, col1]):
        df.loc[i, col1] = df.loc[i, activo] - df.loc[i, col2]

    if pd.isna(df.loc[i, col2]):
        df.loc[i, col2] = df.loc[i, activo] - df.loc[i, col1]

    if pd.isna(df.loc[i, activo]):
        df.loc[i, activo] = df.loc[i, col1] + df.loc[i, col2]


In [ ]:

#PASIVO FIJO
pasivo = "Pasivo fijo\nmil EUR\nÚlt. año disp."
col1 = "Acreedores a L. P.\nmil EUR\nÚlt. año disp."
col2 = "Otros pasivos fijos\nmil EUR\nÚlt. año disp."
col3 = "Provisiones\nmil EUR\nÚlt. año disp."


variables = [col1,col2,col3, pasivo]
for i in df.index:


    if df.loc[i, variables].isna().sum() > 1:
        continue
    #uso pd.isna porque con el loc me da un valor unico y isna no se puede con eso
    if pd.isna(df.loc[i, col1]):
        df.loc[i, col1] = df.loc[i, pasivo] - df.loc[i, col2] - df.loc[i, col3]


    if pd.isna(df.loc[i, col2]):
        df.loc[i, col2] = df.loc[i, pasivo] - df.loc[i, col1] - df.loc[i, col3]


    if pd.isna(df.loc[i, col3]):
        df.loc[i, col3] = df.loc[i, pasivo] - df.loc[i, col1] - df.loc[i, col2]


    if pd.isna(df.loc[i, pasivo]):
        df.loc[i, pasivo] = df.loc[i, col1] + df.loc[i, col2] + df.loc[i, col3]


In [ ]:

#Pasivo líquido:
activo="Pasivo líquido\nmil EUR\nÚlt. año disp."
col1="Deudas financieras\nmil EUR\nÚlt. año disp."
col2="Acreedores comerciales\nmil EUR\nÚlt. año disp."
col3="Otros pasivos líquidos\nmil EUR\nÚlt. año disp."

variables= [col1,col2,col3,activo]

for i in df.index:

    if df.loc[i, variables].isna().sum() > 1:
        continue
    #uso pd.isna porque con el loc me da un valor unico y isna no se puede con eso
    if pd.isna(df.loc[i, col1]):
        df.loc[i, col1] = df.loc[i, activo] - df.loc[i, col2] - df.loc[i, col3]

    if pd.isna(df.loc[i, col2]):
        df.loc[i, col2] = df.loc[i, activo] - df.loc[i, col1] - df.loc[i, col3]

    if pd.isna(df.loc[i, col3]):
        df.loc[i, col3] = df.loc[i, activo] - df.loc[i, col1] - df.loc[i, col2]

    if pd.isna(df.loc[i, activo]):
        df.loc[i, activo] = df.loc[i, col1] + df.loc[i, col2] + df.loc[i, col3]



In [ ]:
#TOTAL PASIVO Y CAPITAL PROPIO

activo="Total pasivo y capital propio\nmil EUR\nÚlt. año disp."
col1="Total activo\nmil EUR\nÚlt. año disp."
col2="Capital suscrito\nmil EUR\nÚlt. año disp."

variables= [col1,col2,activo]

for i in df.index:

    if df.loc[i, variables].isna().sum() > 1:
        continue
    #uso pd.isna porque con el loc me da un valor unico y isna no se puede con eso
    if pd.isna(df.loc[i, col1]):
        df.loc[i, col1] = df.loc[i, activo] + df.loc[i, col2]

    if pd.isna(df.loc[i, col2]):
        df.loc[i, col2] = df.loc[i, col1]-df.loc[i, activo]  

    if pd.isna(df.loc[i, activo]):
        df.loc[i, activo] = df.loc[i, col1] - df.loc[i, col2]


In [ ]:

#RESULTADO BRUTO

activo="Resultado bruto\nmil EUR\nÚlt. año disp."
col1="Importe neto Cifra de Ventas\nmil EUR\nÚlt. año disp."
col2="Consumo de mercaderías y de materias\nmil EUR\nÚlt. año disp."

variables= [col1,col2,activo]

for i in df.index:

    if df.loc[i, variables].isna().sum() > 1:
        continue
    
    if pd.isna(df.loc[i, col1]):
        df.loc[i, col1] = df.loc[i, activo] + df.loc[i, col2]

    if pd.isna(df.loc[i, col2]):
        df.loc[i, col2] = df.loc[i, col1]-df.loc[i, activo]  

    if pd.isna(df.loc[i, activo]):
        df.loc[i, activo] = df.loc[i, col1] - df.loc[i, col2]


In [ ]:

#Resultado explotación:
r_explotacion="Resultado Explotación\nmil EUR\nÚlt. año disp."
col1="Resultado bruto\nmil EUR\nÚlt. año disp."
col2="Otros gastos de explotación\nmil EUR\nÚlt. año disp."

variables= [col1,col2,r_explotacion]
for i in df.index:

    if df.loc[i, variables].isna().sum() > 1:
        continue   
    if pd.isna(df.loc[i, col1]):
        df.loc[i, col1] = df.loc[i, r_explotacion] + df.loc[i, col2]

    if pd.isna(df.loc[i, col2]):
        df.loc[i, col2] = df.loc[i, col1] - df.loc[i, r_explotacion]

    if pd.isna(df.loc[i, r_explotacion]):
        df.loc[i, r_explotacion] = df.loc[i, col1] - df.loc[i, col2]



In [ ]:

#Resultado financiero:
activo="Resultado financiero\nmil EUR\nÚlt. año disp."
col1="Ingresos financieros\nmil EUR\nÚlt. año disp."
col2="Gastos financieros\nmil EUR\nÚlt. año disp."

variables= [col1,col2,activo]

for i in df.index:

    if df.loc[i, variables].isna().sum() > 1:
        continue

    if pd.isna(df.loc[i, col1]):
        df.loc[i, col1] = df.loc[i, activo] - df.loc[i, col2]
        
    if pd.isna(df.loc[i, col2]):
        df.loc[i, col2] = df.loc[i, activo] + df.loc[i, col1]
    
    if pd.isna(df.loc[i, activo]):
        df.loc[i, activo] = df.loc[i, col1] - df.loc[i, col2]



In [ ]:

#RESULTADOS ORDINARIOS ANTES DE IMPUESTOS
pasivo = "Pasivo fijo\nmil EUR\nÚlt. año disp."
col1 = 'Resultado Explotación\nmil EUR\nÚlt. año disp.'
col2 = "Resultado financiero\nmil EUR\nÚlt. año disp."

variables = [col1,col2, pasivo]
for i in df.index:

    if df.loc[i, variables].isna().sum() > 1:
        continue
    
    if pd.isna(df.loc[i, col1]):
        df.loc[i, col1] = df.loc[i, pasivo] - df.loc[i, col2]

    if pd.isna(df.loc[i, col2]):
        df.loc[i, col2] = df.loc[i, pasivo] - df.loc[i, col1]

    if pd.isna(df.loc[i, pasivo]):
        df.loc[i, pasivo] = df.loc[i, col1] + df.loc[i, col2]


In [ ]:

# RESULTADOS ORDINARIOS ANTES DE IMPUESTOS

pasivo = 'Resultado Actividades Ordinarias\nmil EUR\nÚlt. año disp.'
col1 = 'Result. ordinarios antes Impuestos\nmil EUR\nÚlt. año disp.'
col2 = 'Impuestos sobre sociedades\nmil EUR\nÚlt. año disp.'


variables= [col1, col2, pasivo]

for i in df.index:
    
    if df.loc[i, variables].isna().sum() > 1:
        continue

    # Calcular resultado actividades ordinarias (col1)
    if pd.isna(df.loc[i, col1]):
        df.loc[i, col1] = df.loc[i, pasivo] + df.loc[i, col2]

    # Calcular impuestos (col2)
    if  pd.isna(df.loc[i, col2]):
     df.loc[i, col2] = df.loc[i, col1] - df.loc[i, pasivo]

     if pd.isna(df.loc[i, pasivo]):
        df.loc[i, pasivo] = df.loc[i, col1] - df.loc[i, col2]



In [ ]:

# RESULTADOS DE EJERCICIO

pasivo = 'Resultado del Ejercicio\nmil EUR\nÚlt. año disp.'
col1= 'Dotaciones para amortiz. de inmovil.\nmil EUR\nÚlt. año disp.'
col2 = 'Resultado Actividades Ordinarias\nmil EUR\nÚlt. año disp.'


variables= [col1, col2, pasivo]

for i in df.index:
    
    if df.loc[i, variables].isna().sum() > 1:
        continue

    # Calcular resultado actividades ordinarias (col1)
    if pd.isna(df.loc[i, col1]):
        df.loc[i, col1] = df.loc[i, pasivo] - df.loc[i, col2]

    # Calcular impuestos (col2)
    if  pd.isna(df.loc[i, col2]):
     df.loc[i, col2] = df.loc[i, pasivo] - df.loc[i, col1]

     if pd.isna(df.loc[i, pasivo]):
        df.loc[i, pasivo] = df.loc[i, col1] + df.loc[i, col2]


In [ ]:

#EBITDA
ebitda="EBITDA\nmil EUR\nÚlt. año disp."
col1="Resultado Explotación\nmil EUR\nÚlt. año disp."
col2="Dotaciones para amortiz. de inmovil.\nmil EUR\nÚlt. año disp."

variables= [col1,col2,ebitda]
for i in df.index:

    if df.loc[i, variables].isna().sum() > 1:
        continue
   
    if pd.isna(df.loc[i, col1]):
        df.loc[i, col1] = df.loc[i, ebitda] - df.loc[i, col2]

    if pd.isna(df.loc[i, col2]):
        df.loc[i, col2] = df.loc[i, ebitda] - df.loc[i, col1]

    if pd.isna(df.loc[i, ebitda]):
        df.loc[i, ebitda] = df.loc[i, col1] + df.loc[i, col2]



In [ ]:
df.isna().mean()*100

## IMPUTAR (info externa, 0, mediana y por sector)

In [ ]:
df = df.drop(["Ingresos extraordinarios\nmil EUR\nÚlt. año disp.", "Gastos extraordinarios\nmil EUR\nÚlt. año disp.", "Resultados actividades extraordinarias\nmil EUR\nÚlt. año disp."], axis=1)
#dropeo las columnas con 100% o 99% de nulos porque son extraordinarias y no me aporta info

In [ ]:
# rellenamos los nulos que se pueden (son pocos y están en internet)

# 1. en localidad
df[df["Localidad"].isna()]

In [ ]:
df.loc[df["Localidad"].isna(), "Localidad"] = "Deba"
df[df["Localidad"].isna()]

In [ ]:
#2. en fecha
df[df["Fecha constitución"].isna()]

In [ ]:
fechas = [
    "24-11-1846",
    "23-03-2021",
    "22-07-1991",
    "19-11-2014",
    "22-02-2017",
    "18-01-1962"
]  #estas fechas las hemos buscado en internet (correspondientes a cada empresa)

nulos = df["Fecha constitución"].isna()
df.loc[nulos, "Fecha constitución"] = fechas[:nulos.sum()]  


In [ ]:
df[df["Fecha constitución"].isna()]

In [ ]:
df.isna().mean()*100

In [ ]:
# hay columnas que tiene sentido que sean 0 y por eso no tienen información disponible. 
# por eso sustituimos sus nas por 0. nos hemos fijado en el porcentaje de nas que tenian las columnas.
    # acreedores a lp y provisiones parece que hay muchos no reportados por la cantidad de nulos que hay
cols_cero = [
"Acreedores a L. P.\nmil EUR\nÚlt. año disp.",
"Otros pasivos líquidos\nmil EUR\nÚlt. año disp.",
"Otros pasivos fijos\nmil EUR\nÚlt. año disp.",
"Provisiones\nmil EUR\nÚlt. año disp.",
"Ingresos financieros\nmil EUR\nÚlt. año disp.",
"Gastos financieros\nmil EUR\nÚlt. año disp.",
"Consumo de mercaderías y de materias\nmil EUR\nÚlt. año disp.",
"Otros gastos de explotación\nmil EUR\nÚlt. año disp.",
"Dotaciones para amortiz. de inmovil.\nmil EUR\nÚlt. año disp."
]

df[cols_cero] = df[cols_cero].fillna(0)

df.isna().mean()*100

separar por cnae para imputar por mediana según sector:


In [ ]:
df["sector"] = df["Código primario CNAE 2009"].astype(str).str[:2]
df.head()

In [ ]:
df["sector"].unique()

In [ ]:
#lo primero que vamos a hacer es crear una variable con todas las columnas que queremos imputar

columnas = ["Existencias\nmil EUR\nÚlt. año disp.","Inmovilizado\nmil EUR\nÚlt. año disp.",

"Inmovilizado inmaterial\nmil EUR\nÚlt. año disp.",

"Inmovilizado material\nmil EUR\nÚlt. año disp.",

"Otros activos fijos\nmil EUR\nÚlt. año disp.",

"Activo circulante\nmil EUR\nÚlt. año disp.",

"Acreedores a L. P.\nmil EUR\nÚlt. año disp.",

"Deudores\nmil EUR\nÚlt. año disp.",

"Otros activos líquidos\nmil EUR\nÚlt. año disp.",

"Total activo\nmil EUR\nÚlt. año disp.",

"Fondos propios\nmil EUR\nÚlt. año disp.",

"Capital suscrito\nmil EUR\nÚlt. año disp.",

"Otros fondos propios\nmil EUR\nÚlt. año disp.",

"Pasivo fijo\nmil EUR\nÚlt. año disp.",

"Otros pasivos fijos\nmil EUR\nÚlt. año disp.",

"Provisiones\nmil EUR\nÚlt. año disp.",

"Pasivo líquido\nmil EUR\nÚlt. año disp.",

"Acreedores comerciales\nmil EUR\nÚlt. año disp.",

"Otros pasivos líquidos\nmil EUR\nÚlt. año disp.",

"Total pasivo y capital propio\nmil EUR\nÚlt. año disp.",

"Número empleados\nÚlt. año disp.",

"Importe neto Cifra de Ventas\nmil EUR\nÚlt. año disp.",

"Consumo de mercaderías y de materias\nmil EUR\nÚlt. año disp.",

"Resultado bruto\nmil EUR\nÚlt. año disp.",

"Otros gastos de explotación\nmil EUR\nÚlt. año disp.",

"Resultado Explotación\nmil EUR\nÚlt. año disp.",

"Ingresos financieros\nmil EUR\nÚlt. año disp.",

"Gastos financieros\nmil EUR\nÚlt. año disp.",

"Resultado financiero\nmil EUR\nÚlt. año disp.",

"Result. ordinarios antes Impuestos\nmil EUR\nÚlt. año disp.",

"Impuestos sobre sociedades\nmil EUR\nÚlt. año disp.",

"Resultado Actividades Ordinarias\nmil EUR\nÚlt. año disp.",

"Resultado del Ejercicio\nmil EUR\nÚlt. año disp.",

"Dotaciones para amortiz. de inmovil.\nmil EUR\nÚlt. año disp.",

"sector"]


In [ ]:
df.dtypes

In [ ]:
for col in columnas:

    df[col] = df[col].astype(float)

    medianas = df.groupby("sector")[col].median()

    df[col] = df[col].fillna(df["sector"].map(medianas))



In [ ]:
df.isna().mean()*100

## RANDOM FOREST

In [ ]:
columnas2=df[["Deudas financieras\nmil EUR\nÚlt. año disp.", "EBITDA\nmil EUR\nÚlt. año disp.","Tesorería\nmil EUR\nÚlt. año disp."]]
#como estas consideramos que son variables importantes queremos mas precision --> random forest

In [ ]:
#lo primero de todo es realizar una matriz de correlacion para ver las relaciones entre las variables del dataframe con las que vamos a querer imputar


plt.figure(figsize=(10,8))
sns.heatmap(df.corr(numeric_only=True), cmap="coolwarm", center=0)
plt.show()

In [ ]:
#como la matriz de correlacion no nos muestra nada vamos a ir mirando las correlaciones de las columnas que queremos calcular uno por una
df.corr(numeric_only=True)["EBITDA\nmil EUR\nÚlt. año disp."].sort_values(ascending=False).head(10)

In [ ]:
df.corr(numeric_only=True)["Deudas financieras\nmil EUR\nÚlt. año disp."].sort_values(ascending=False).head(10)

In [ ]:
df.corr(numeric_only=True)["Tesorería\nmil EUR\nÚlt. año disp."].sort_values(ascending=False).head(10)

In [ ]:
#ahora vamos a realizar los modelos random forest para terminar de imputar todos los datos

In [ ]:
train = df[df["EBITDA\nmil EUR\nÚlt. año disp."].notna()]
test = df[df["EBITDA\nmil EUR\nÚlt. año disp."].isna()]

In [ ]:
df_train = df[df["EBITDA\nmil EUR\nÚlt. año disp."].notna()]
df_test = df[df["EBITDA\nmil EUR\nÚlt. año disp."].isna()]

X_train = df_train[["Resultado Explotación\nmil EUR\nÚlt. año disp.", "Fondos propios\nmil EUR\nÚlt. año disp.", "Inmovilizado\nmil EUR\nÚlt. año disp."]]
y_train = df_train["EBITDA\nmil EUR\nÚlt. año disp."]

X_test = test[["Resultado Explotación\nmil EUR\nÚlt. año disp.", "Fondos propios\nmil EUR\nÚlt. año disp.", "Inmovilizado\nmil EUR\nÚlt. año disp."]]

In [ ]:
rf = RandomForestRegressor()
rf.fit(X_train, y_train)

In [ ]:
df.loc[df["EBITDA\nmil EUR\nÚlt. año disp."].isna(), "EBITDA\nmil EUR\nÚlt. año disp."] = rf.predict(X_test)

In [ ]:
df["EBITDA\nmil EUR\nÚlt. año disp."].isna().mean()*100

In [ ]:
train = df[df["Deudas financieras\nmil EUR\nÚlt. año disp."].notna()]
test = df[df["Deudas financieras\nmil EUR\nÚlt. año disp."].isna()]

In [ ]:
df_train = df[df["Deudas financieras\nmil EUR\nÚlt. año disp."].notna()]
df_test = df[df["Deudas financieras\nmil EUR\nÚlt. año disp."].isna()]

X_train = df_train[["Pasivo fijo\nmil EUR\nÚlt. año disp.","Acreedores a L. P.\nmil EUR\nÚlt. año disp." ,"Total activo\nmil EUR\nÚlt. año disp." ]]
y_train = df_train["Deudas financieras\nmil EUR\nÚlt. año disp."]

X_test = test[["Pasivo fijo\nmil EUR\nÚlt. año disp.","Acreedores a L. P.\nmil EUR\nÚlt. año disp." ,"Total activo\nmil EUR\nÚlt. año disp." ]]

In [ ]:
rf = RandomForestRegressor()
rf.fit(X_train, y_train)

In [ ]:
df.loc[df["Deudas financieras\nmil EUR\nÚlt. año disp."].isna(), "Deudas financieras\nmil EUR\nÚlt. año disp."] = rf.predict(X_test)

In [ ]:
df["Deudas financieras\nmil EUR\nÚlt. año disp."].isna().mean()*100

In [ ]:
# como tesorería no tiene mucha correlación con ninguna, imputamos con la mediana

In [ ]:
df["Tesorería\nmil EUR\nÚlt. año disp."] = df["Tesorería\nmil EUR\nÚlt. año disp."].astype(float)

medianas = df.groupby("sector")[col].median()

df["Tesorería\nmil EUR\nÚlt. año disp."] = df["Tesorería\nmil EUR\nÚlt. año disp."].fillna(df["sector"].map(medianas))

In [ ]:
df.isna().mean()*100

In [ ]:
df.to_csv("df_limpio.csv", index=False)